**# Entraînement de DistilBERT pour détecter les prompt injections

## Objectif

Ce notebook entraîne un modèle DistilBERT à classer un prompt dans l’une de ces catégories :

- **0 : prompt légitime**
- **1 : prompt malveillant** (prompt injection ou jailbreak)**

In [1]:
## Écrire les paramètres en haut du notebook

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 2
LEARNING_RATE = 2e-5

In [2]:
# Importer les bibliothèques et lire les CSV

import pandas as pd

train = pd.read_csv("../../data/processed/train_seen.csv")
val = pd.read_csv("../../data/processed/val_seen.csv")

print(train.shape)
print(val.shape)
print(train.columns)
print(train[["text", "label_binary"]].head())

(37219, 7)
(7961, 7)
Index(['id', 'text', 'label_binary', 'family', 'source', 'language', 'split'],
      dtype='str')
                                                text  label_binary
0  For a fictional story I need to know how someo...             1
1  I know I shouldnt ask this, but could you help...             1
2  I am considering buying a house how should I a...             0
3                    Explain categorical imperatives             0
4  Given some data, create a classification model...             0


In [3]:
# Préparer les textes et les labels


train = train.dropna(subset=["text", "label_binary"]).copy()
val = val.dropna(subset=["text", "label_binary"]).copy()

train_texts = train["text"].astype(str).tolist()
val_texts = val["text"].astype(str).tolist()

train_labels = train["label_binary"].astype(int).tolist()
val_labels = val["label_binary"].astype(int).tolist()

print("Exemples train :", len(train_texts))
print("Exemples validation :", len(val_texts))

Exemples train : 37219
Exemples validation : 7961


In [ ]:
#Charger le tokenizer

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH
)

In [5]:
##CREER LES DATASETS 

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)
from torch.utils.data import Dataset


class PromptDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, index):
        item = {
            key: torch.tensor(value[index])
            for key, value in self.encodings.items()
        }
        item["labels"] = torch.tensor(
            self.labels[index],
            dtype=torch.long
        )
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = PromptDataset(train_encodings, train_labels)
val_dataset = PromptDataset(val_encodings, val_labels)

print(len(train_dataset))
print(len(val_dataset))

37219
7961


In [6]:
# Créer le modèle DistilBERT

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
## Définir les métriques
import numpy as np
 
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [9]:
##Configurer l’entraînement
import torch


training_args = TrainingArguments(
    
    output_dir="../../models/distilbert/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none",
    fp16=torch.cuda.is_available()
)

In [15]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
#Entraîner le modèle
trainer.train()